In [0]:
-- Buscar papers relevantes por similitud semantica al objetivo.
-- goal_embedding ya viene calculado desde Python (Fase 3, mismo modelo que los papers).
CREATE OR REPLACE FUNCTION search_papers(
  goal_embedding vector(384),
  top_k INT DEFAULT 6
)
RETURNS TABLE (paper_id TEXT, title TEXT, abstract TEXT, doi TEXT, oa_url TEXT)
LANGUAGE SQL
AS $$
  SELECT paper_id, title, abstract, doi, oa_url
  FROM papers
  WHERE embedding IS NOT NULL
  ORDER BY embedding <=> goal_embedding
  LIMIT top_k;
$$;
 
-- Anadir un paper a una coleccion
CREATE OR REPLACE FUNCTION add_to_collection(
  p_collection_id UUID,
  p_paper_id TEXT
)
RETURNS TEXT
LANGUAGE SQL
AS $$
  INSERT INTO collection_papers (collection_id, paper_id)
  VALUES (p_collection_id, p_paper_id)
  ON CONFLICT DO NOTHING
  RETURNING 'Anadido: ' || paper_id;
$$;
 
-- Siguiente paper recomendado segun progreso
CREATE OR REPLACE FUNCTION get_next_paper(
  p_user_id UUID,
  p_collection_id UUID
)
RETURNS TABLE (paper_id TEXT, title TEXT)
LANGUAGE SQL
AS $$
  SELECT p.paper_id, p.title
  FROM collection_papers cp
  JOIN papers p ON p.paper_id = cp.paper_id
  LEFT JOIN reading_progress rp
    ON rp.paper_id = p.paper_id AND rp.user_id = p_user_id
  WHERE cp.collection_id = p_collection_id
    AND (rp.status IS NULL OR rp.status != 'done')
  ORDER BY p.publication_year ASC
  LIMIT 1;
$$;
